In [ ]:
import zipfile
import random
import pandas as pd


## Analyse rapide des données brutes

Pour l'instant, on a uniquement les fichiers txt de l'élection de 1981 mais on a les métadonnées pour tout
Il faudra adapter pour intégrer toutes les années

In [9]:
#lecture des métadonnées
metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")

/tmp/ipykernel_8336/4037180946.py:2: DtypeWarning: Columns (8,9,10,12,28,29,30,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")


In [32]:
zip_path = "data/legislatives.zip"

all_texts = []

with zipfile.ZipFile(zip_path, "r") as z:
    
    # on liste les fichiers txt
    txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    
    # on filtre les dossiers parasites type __MACOSX
    txt_filtres = [f for f in txt_files if not f.startswith("__MACOSX")]
    
    print(f"Nombre de fichiers txt valides : {len(txt_filtres)}")

    for file in txt_filtres:
        try:
            with z.open(file) as f:
                text_content = f.read().decode("utf-8")
                # garder uniquement le nom du fichier (sans dossier)
                filename = file.split("/")[-1].replace(".txt", "")
                print(filename)
                
                all_texts.append({
                    "id": filename,
                    "text": text_content
                })
        
        except Exception as e:
            print(f"Erreur lors de la lecture de {file} : {e}")

# --- DataFrame textes ---
all_texts_df = pd.DataFrame(all_texts)

# --- Fusion texte + métadonnées ---
meta_et_texts = metadonnees.merge(all_texts_df, on="id", how="right")

# --- Vérification des correspondances ---
missing_meta = meta_et_texts[meta_et_texts["text"].isnull()]["id"]

if len(missing_meta) > 0:
    print("⚠️ Certains fichiers n'ont pas de texte associé :")
    print(missing_meta.tolist())

print(meta_et_texts.head())
print(f"Nombre total de documents : {len(meta_et_texts)}")


Nombre de fichiers txt valides : 3182
EL136_L_1981_06_075_12_1_PF_03
EL134_L_1981_06_035_01_1_PF_01
EL135_L_1981_06_057_06_1_PF_04
EL136_L_1981_06_067_07_1_PF_02
EL134_L_1981_06_013_06_1_PF_02
EL134_L_1981_06_006_06_1_PF_02
EL137_L_1981_06_988_02_1_PF_07
EL136_L_1981_06_069_02_1_PF_01
EL134_L_1981_06_008_03_1_PF_01
EL134_L_1981_06_033_04_1_PF_03
EL135_L_1981_06_036_02_2_PF_02
EL135_L_1981_06_059_21_1_PF_01
EL137_L_1981_06_076_03_1_PF_06
EL135_L_1981_06_038_07_2_PF_01
EL137_L_1981_06_094_06_1_PF_02
EL136_L_1981_06_075_30_1_PF_05
EL135_L_1981_06_054_05_1_PF_01
EL137_L_1981_06_078_08_2_PF_02
EL136_L_1981_06_067_01_1_BV_pdfmasterocr
EL137_L_1981_06_078_06_1_PF_05
EL137_L_1981_06_093_06_1_PF_01
EL134_L_1981_06_017_05_1_PF_04
EL134_L_1981_06_002_05_1_PF_04
EL135_L_1981_06_060_02_1_PF_01
EL137_L_1981_06_078_01_2_PF_01
EL134_L_1981_06_018_02_2_PF_02
EL137_L_1981_06_088_03_1_PF_02
EL137_L_1981_06_076_04_2_PF_02
EL137_L_1981_06_095_03_1_PF_03
EL137_L_1981_06_080_03_1_PF_03
EL135_L_1981_06_059_16

In [33]:
meta_et_texts.head()

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,text
0,EL136_L_1981_06_075_12_1_PF_03,1981-06-14,Ve République;France;Élections législatives;As...,"Élections législatives de 1981, Paris - 75, ci...",législatives,1.0,EL136,75,Paris,75 - Paris (Seine),...,entre 30 et 39 ans,employé PTT,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Sciences Po / fonds CEVIPOF\nElections législa...
1,EL134_L_1981_06_035_01_1_PF_01,1981-06-14,France;Assemblée Nationale;Élections législati...,"Élections législatives de 1981, Ille-et-Vilain...",législatives,1.0,EL134,35,Ille-et-Vilaine,35 - Ille-et-Vilaine,...,entre 20 et 29 ans,institutrice,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Sciences Po / fonds CEVIPOF\nElections législa...
2,EL135_L_1981_06_057_06_1_PF_04,1981-06-14,Assemblée Nationale;France;Élections législati...,"Élections législatives de 1981, Moselle - 57, ...",législatives,1.0,EL135,57,Moselle,57 - Moselle,...,entre 20 et 29 ans,surveillant internat,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Élections législatives de juin 1981 6e circons...
3,EL136_L_1981_06_067_07_1_PF_02,1981-06-14,France;Ve République;Assemblée Nationale;Élect...,"Élections législatives de 1981, Bas-Rhin - 67,...",législatives,1.0,EL136,67,Bas-Rhin,67 - Bas-Rhin,...,non mentionné,ouvrier spécialisé,non mentionné,non mentionné,non mentionné,non mentionné,Parti socialiste,non mentionné,non,Sciences Po / fonds CEVIPOF\na\nELECTIONS LEGI...
4,EL134_L_1981_06_013_06_1_PF_02,1981-06-14,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1981, Bouches-du-Rhô...",législatives,1.0,EL134,13,Bouches-du-Rhône,13 - Bouches-du-Rhône,...,non mentionné,employée,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,Union de la gauche,non,RÉPUBLIQUE FRANÇAISE · LIBERTÉ ÉGALITÉ FRATERN...


## Pre-traitement des données

## Représentation textuelle

## Classification par thème

## Croisement avec les professions des candidats

## Analyses statistiques diverses